In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"

In [3]:
from concept_abstraction.selection import *
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch.nn as nn
from torchvision import models
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
import pickle 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
is_jupyter = 'ipykernel' in sys.modules

In [140]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 121
    out_folder = "cub"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [141]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 121}


In [142]:
np.random.seed(seed)
random.seed(seed)

In [143]:
def get_performance(selected_concepts,accuracy_by_concept):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

## Perfrect Concepts

In [144]:
results['perfect'] = {}
results['imperfect'] = {}
results['intervention'] = {}

In [145]:
train = pickle.load(open("../../data/cub/train.pkl","rb"))
test = pickle.load(open("../../data/cub/test.pkl","rb"))

In [146]:
train_X = np.array([i['attribute_label'] for i in train])
train_Y = np.array([i['class_label'] for i in train])
test_X = np.array([i['attribute_label'] for i in test])
test_Y = np.array([i['class_label'] for i in test])

In [147]:
def get_performance(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc


#### Imperfect Concepts

In [148]:
def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(256),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(pred_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [149]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

train = pickle.load(open("../../data/cub/train_error.pkl","rb"))
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))

pred_train_X = sigmoid(np.array([i['attribute_label'] for i in train])).round()
train_Y = np.array([i['class_label'] for i in train])
pred_test_X = sigmoid(np.array([i['attribute_label'] for i in test])).round()
test_Y = np.array([i['class_label'] for i in test])

In [150]:
num_concepts_range = range(20,num_concepts_selected,20)

In [151]:
train_concept_accuracy = np.mean(pred_train_X.round() == train_X,axis=0)
test_concept_accuracy = np.mean(pred_test_X.round() == test_X,axis=0)

In [152]:
manually_selected_concepts = open("../../data/cub/manual_concepts.txt").read().strip().split("\n")
manually_selected_concepts = [int(i) for i in manually_selected_concepts]


In [153]:
results['imperfect']['manual'] = {'reward': get_performance_real(manually_selected_concepts), 'concepts': manually_selected_concepts}

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [41]:
results['imperfect']['random'] = {}

for c in num_concepts_range:
    random_concepts = random.sample(list(range(312)),c)
    results['imperfect']['random'][c] = {
        'reward': get_performance_real(random_concepts), 
        'concepts': random_concepts
    }
results['imperfect']['random']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.15119088712461168,
  'concepts': [57,
   12,
   140,
   125,
   114,
   71,
   52,
   279,
   44,
   302,
   216,
   16,
   15,
   47,
   111,
   119,
   258,
   308,
   13,
   287]}}

In [45]:
results['imperfect']['entropy'] = {}

for c in num_concepts_range:
    entropy_concepts = basic_greedy_selection_supervised(train_X,train_Y,c)
    results['imperfect']['entropy'][c] = {
        'reward': get_performance_real(entropy_concepts), 
        'concepts': entropy_concepts
    }
results['imperfect']['entropy']

[0.24998083 0.24995266 0.24560361 0.24473498 0.24395134 0.24265612
 0.24214729 0.241924   0.2411571  0.2411571  0.23769547 0.2339816
 0.23339578 0.2330718  0.23246971 0.22802237 0.22429046 0.22422355
 0.2238877  0.21352187] 0.0543086312740497


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.6037279944770452,
  'concepts': [236,
   289,
   240,
   51,
   212,
   20,
   35,
   209,
   235,
   6,
   178,
   218,
   132,
   117,
   151,
   54,
   163,
   149,
   304,
   90]}}

In [46]:
results['imperfect']['greedy'] = {}

for c in num_concepts_range:
    greedy_concepts = greedy_selection_supervised(train_X,train_Y,c)
    results['imperfect']['greedy'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['greedy']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


{20: {'reward': 0.40317569899896444,
  'concepts': [220,
   235,
   6,
   4,
   230,
   274,
   131,
   101,
   119,
   203,
   254,
   126,
   173,
   158,
   116,
   15,
   45,
   259,
   111,
   30]}}

In [163]:
results['imperfect']['lp_hybrid'] = {}

for c in num_concepts_range:
    greedy_concepts = lp_selection_supervised(train_X,train_Y,c)
    results['imperfect']['lp_hybrid'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['lp_hybrid']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

{20: {'reward': 0.5367621677597515,
  'concepts': [6,
   20,
   35,
   51,
   54,
   90,
   117,
   132,
   149,
   151,
   163,
   178,
   209,
   212,
   218,
   235,
   236,
   240,
   289,
   304]},
 40: {'reward': 0.5931998619261305,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   29,
   35,
   51,
   54,
   59,
   63,
   69,
   75,
   90,
   101,
   117,
   132,
   149,
   151,
   163,
   178,
   193,
   209,
   212,
   218,
   235,
   236,
   240,
   244,
   253,
   259,
   260,
   268,
   274,
   289,
   304,
   308,
   311]},
 60: {'reward': 0.608560579910252,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   29,
   35,
   36,
   45,
   50,
   51,
   53,
   54,
   59,
   63,
   69,
   75,
   80,
   84,
   90,
   101,
   111,
   116,
   117,
   131,
   132,
   149,
   151,
   163,
   168,
   172,
   178,
   179,
   187,
   193,
   194,
   203,
   209,
   212,
   213,
   218,
   220,
   235,
   236,
   240,
   244,
   249,
   253,
   259,
   260,
   268

In [193]:
results['imperfect']['multiple_log'] = {}

for c in num_concepts_range:
    greedy_concepts = multiple_log_selection_supervised(train_X,train_Y,train_concept_accuracy,c)
    results['imperfect']['multiple_log'][c] = {
        'reward': get_performance_real(greedy_concepts), 
        'concepts': greedy_concepts
    }
results['imperfect']['multiple_log']

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

{20: {'reward': 0.535036244390749,
  'concepts': [6,
   20,
   35,
   51,
   54,
   90,
   117,
   132,
   149,
   151,
   163,
   178,
   209,
   212,
   218,
   235,
   236,
   240,
   289,
   304]},
 40: {'reward': 0.5923369002416292,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   29,
   35,
   51,
   54,
   59,
   63,
   69,
   75,
   90,
   101,
   117,
   132,
   149,
   151,
   163,
   178,
   193,
   209,
   212,
   218,
   235,
   236,
   240,
   244,
   253,
   259,
   260,
   268,
   274,
   289,
   304,
   308,
   311]},
 60: {'reward': 0.6102865032792544,
  'concepts': [6,
   7,
   10,
   14,
   20,
   21,
   25,
   29,
   35,
   36,
   45,
   50,
   51,
   53,
   54,
   59,
   63,
   69,
   75,
   80,
   84,
   90,
   101,
   111,
   116,
   117,
   131,
   132,
   149,
   151,
   163,
   168,
   172,
   178,
   179,
   187,
   193,
   194,
   203,
   209,
   212,
   213,
   218,
   220,
   235,
   236,
   240,
   244,
   249,
   253,
   259,
   260,
   268

## Intervention

In [84]:
test = pickle.load(open("../../data/cub/test_error.pkl","rb"))


In [114]:
num_concepts = len(manually_selected_concepts)
manual_selection = manually_selected_concepts
random_selection = random.sample(list(range(312)),112)
entropy_selection = basic_greedy_selection_supervised(train_X,train_Y,112)
greedy_selection = greedy_selection_supervised(train_X,train_Y,112)
lp_selection  = lp_selection_supervised(train_X,train_Y,112)
multiple_selection  = multiple_log_selection_supervised(train_X,train_Y,train_concept_accuracy,112)


In [115]:
def get_performance_real(selected_concepts):
    mlp = MLPClassifier(
        hidden_layer_sizes=(256),
        activation='relu',
        solver='adam',
    )

    # Train the model
    mlp.fit(pred_train_X[:,selected_concepts], train_Y)

    # Predict on the test set
    y_pred = mlp.predict(intervention_test_X[:,selected_concepts])

    # Compute accuracy
    acc = accuracy_score(test_Y.reshape(-1,1), y_pred)
    return acc

In [107]:
results['intervention'] = {}

In [137]:
intervention_percent = 0.2
arr = lp_selection
description = "lp_hybrid"
num_cols = test_X.shape[1]
num_cols_to_intervene = int((1-intervention_percent) * num_cols)

# Randomly pick columns
cols = np.random.choice(
    num_cols, num_cols_to_intervene, replace=False
)

# Start from original
intervention_test_X = test_X.copy()

# Replace selected columns entirely
intervention_test_X[:, cols] = pred_test_X[:, cols]
results['intervention'][description] = {}
results['intervention'][description][intervention_percent] = {
            'reward': get_performance_real(arr)
        }

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [108]:
for intervention_percent in [0.2,0.4,0.6,0.8,1.0]:
    num_cols = test_X.shape[1]
    num_cols_to_intervene = int((1-intervention_percent) * num_cols)

    # Randomly pick columns
    cols = np.random.choice(
        num_cols, num_cols_to_intervene, replace=False
    )

    # Start from original
    intervention_test_X = test_X.copy()

    # Replace selected columns entirely
    intervention_test_X[:, cols] = pred_test_X[:, cols]

    for arr,description in zip([manually_selected_concepts,
                                lp_selection,
                                multiple_selection,
                                greedy_selection,
                                random_selection,
                                entropy_selection
                                ],[
                                    "manual","lp_hybrid","multiple_log",
                                    'greedy','random',
                                    'entropy'
                                ]):
        if description not in results['intervention']:
            results['intervention'][description] = {}
        results['intervention'][description][intervention_percent] = {
            'reward': get_performance_real(arr)
        }
        print(description,intervention_percent,results['intervention'][description][intervention_percent]['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


manual 0.2 0.11788056610286503


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


lp 0.2 0.12944425267518123


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


greedy 1.0 0.9898170521228857


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


random 1.0 0.9589230238177425
entropy 1.0 1.0


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## Save Data

In [110]:
save_path = get_save_path(out_folder,save_name)

In [111]:
delete_duplicate_results(out_folder,"",results)

In [112]:
json.dump(results,open('../../results/'+save_path,'w'))